# Flicker change blindness: analysis

Computes the measures of Rensink, O'Regan & Clark (1997) from `*_trials.csv` files:

- **alternations to identification** (p. 370): the dependent variable, "proportional to the reaction time", here `rt_ms / alternation_s` (the time from one A↔A' switch to the next); averaged over **correct** trials only, i.e. trials where both the place and the type of change were reported correctly;
- the **identification error rate** (paper: 1.2 % averaged over all experiments, p. 370);
- the comparison of **central vs marginal interest** within each type of change (paper Fig. 3a: marginal slower for every type).

`ANALYSIS_RUN` (or `RUN` below) is a run folder, or any folder above several runs: every `*_trials.csv` under it is read and each subject's trials are averaged first, as the paper's statistics are over observers.

These numbers come from whatever produced the files. A simulated session's observer draws its alternations from the paper's own means (src/rensink1997/simulate.py), so on simulated data the notebook only shows that the analysis runs.

In [ ]:
import os
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use('Agg')
import matplotlib.pyplot as plt

HERE = Path.cwd() if (Path.cwd() / 'run.py').exists() else Path.cwd().parent
RUN = os.environ.get('ANALYSIS_RUN')
if RUN is None:
    runs = sorted(HERE.glob('data*/**/*_trials.csv'), key=lambda p: p.stat().st_mtime)
    RUN = runs[-1].parent if runs else None
print('reading under:', RUN)

In [ ]:
files = sorted(Path(RUN).rglob('*_trials.csv')) if RUN else []
frames = []
for f in files:
    d = pd.read_csv(f)
    d['subject'] = f.name.split('_')[0]
    d['run_file'] = f.name
    frames.append(d)
trials = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(len(files), 'file(s),', len(trials), 'trials,', trials['subject'].nunique() if len(trials) else 0, 'subject(s)')
if len(trials):
    print(trials.groupby('outcome').size().to_string())

## Identification errors and missed changes

In [ ]:
PAPER = {'error_rate_pct': 1.2, 'exp1_central': 7.3, 'exp1_marginal': 17.1, 'no_blank': 1.4}
if len(trials):
    identified = trials[trials['outcome'].isin(['CORRECT', 'MISIDENTIFIED'])]
    err = 100 * (identified['outcome'] == 'MISIDENTIFIED').mean() if len(identified) else float('nan')
    missed = 100 * (trials['outcome'] == 'NO_DETECTION').mean()
    print(f'identification error rate: {err:.1f} % (paper: {PAPER["error_rate_pct"]} %)')
    print(f'no change seen within the time limit: {missed:.1f} % of trials')
    print(f'detected but no report: {100 * (trials["outcome"] == "NO_REPORT").mean():.1f} %')
    if len(identified):
        print('of misidentified trials, wrong place / wrong type:',
              int((~identified['location_correct'].astype(bool)).sum()), '/',
              int((~identified['type_correct'].astype(bool)).sum()))

## Alternations to identification, by interest and type of change (correct trials)

In [ ]:
correct = trials[trials['outcome'] == 'CORRECT'].copy() if len(trials) else pd.DataFrame()
if len(correct):
    correct['seconds'] = correct['rt_ms'] / 1000
    per_subject = (correct.groupby(['subject', 'change_type', 'interest'])['alternations']
                   .mean().rename('alternations').reset_index())
    table = (per_subject.groupby(['change_type', 'interest'])['alternations']
             .agg(['mean', 'sem', 'count']).round(2))
    print('mean alternations per subject, then across subjects (count = subjects):')
    print(table.to_string())
    by_interest = correct.groupby('interest')[['alternations', 'seconds']].mean().round(2)
    print()
    print('pooled over types (trial means):')
    print(by_interest.to_string())
    print(f"paper, Experiment 1: central {PAPER['exp1_central']}, marginal {PAPER['exp1_marginal']} alternations; no blanks {PAPER['no_blank']}")
    print()
    print('by scene source (photo pairs vs procedural pairs):')
    print(correct.groupby(['source', 'interest'])['alternations'].agg(['mean', 'count']).round(2).to_string())

## Central vs marginal within each type of change

The paper tested marginal against central interest within each type of change across its ten observers (p. 370). With two or more subjects this computes a paired comparison of the subject means with an exact sign-flip permutation test (no distributional assumption, no extra dependency). With one subject there is nothing to test across observers.

In [ ]:
def sign_flip_p(diffs):
    diffs = np.asarray(diffs, float)
    n = len(diffs)
    if n == 0:
        return float('nan')
    observed = abs(diffs.mean())
    signs = np.array(np.meshgrid(*[[-1, 1]] * n)).reshape(n, -1).T if n <= 16 else \
        np.random.default_rng(0).choice([-1, 1], size=(20000, n))
    return float((np.abs((signs * diffs).mean(axis=1)) >= observed - 1e-12).mean())

if len(correct):
    wide = per_subject.pivot_table(index=['subject', 'change_type'], columns='interest',
                                   values='alternations').reset_index()
    for ct, g in wide.groupby('change_type'):
        if {'central', 'marginal'} <= set(g.columns):
            g = g.dropna(subset=['central', 'marginal'])
            d = g['marginal'] - g['central']
            if len(d) >= 2:
                print(f'{ct}: marginal - central = {d.mean():.2f} alternations, n = {len(d)} subjects, sign-flip p = {sign_flip_p(d):.4f}')
            else:
                print(f'{ct}: {len(d)} subject(s) with both interests; no test across observers')

## Figure 3a style plot

In [ ]:
if len(correct):
    types = ['presence', 'color', 'location']
    fig, ax = plt.subplots(figsize=(6, 3.6), dpi=110)
    for j, (interest, colour) in enumerate([('central', '#555555'), ('marginal', '#bbbbbb')]):
        means = [table['mean'].get((t, interest), np.nan) for t in types]
        sems = [table['sem'].get((t, interest), np.nan) for t in types]
        ax.bar(np.arange(3) + (j - 0.5) * 0.38, means, 0.36, yerr=np.nan_to_num(sems),
               color=colour, edgecolor='black', label=f'{interest} interest')
    ax.axhline(PAPER['no_blank'], ls='--', color='black', lw=1, label='paper: no blanks')
    ax.set_xticks(range(3), ['Presence/\nabsence', 'Color', 'Location'])
    ax.set_ylabel('alternations')
    ax.legend(frameon=False, fontsize=8)
    ax.set_title('alternations to identification (correct trials)', fontsize=10)
    fig.tight_layout()
    out = Path(RUN) / 'alternations_by_interest.png'
    fig.savefig(out)
    print('saved', out)